In [ ]:
pip install tensorflow numpy matplotlib


In [ ]:
# lstm_text_prediction.py

import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
import os
import matplotlib.pyplot as plt

def main():
    # 1. Download and Read the Dataset
    print("Downloading and reading the dataset...")
    path = tf.keras.utils.get_file('shakespeare.txt',
                                   'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt')
    with open(path, 'rb') as f:
        text = f.read().decode(encoding='utf-8')
    print(f'Length of text: {len(text)} characters')

    # 2. Text Preprocessing
    print("\nPreprocessing the text...")
    vocab = sorted(set(text))
    print(f'Unique characters: {len(vocab)}')

    # Creating mapping from unique characters to indices
    char2idx = {u: i for i, u in enumerate(vocab)}
    idx2char = np.array(vocab)

    # Converting all text into integers
    text_as_int = np.array([char2idx[c] for c in text])
    print(f'Total characters in text: {len(text_as_int)}')

    # 3. Create Training Examples and Targets
    print("\nCreating training sequences...")
    seq_length = 100  # Length of each input sequence
    examples_per_epoch = len(text_as_int) - seq_length

    # Create a dataset of sequences
    char_dataset = tf.data.Dataset.from_tensor_slices(text_as_int)
    sequences = char_dataset.batch(seq_length + 1, drop_remainder=True)

    # Split each sequence into input and target
    def split_input_target(chunk):
        input_text = chunk[:-1]
        target_text = chunk[1:]
        return input_text, target_text

    dataset = sequences.map(split_input_target)

    # 4. Create Training Batches
    BATCH_SIZE = 64
    BUFFER_SIZE = 10000

    dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

    # 5. Define Model Parameters
    vocab_size = len(vocab)  # Number of unique characters
    embedding_dim = 256      # Dimension of embedding layer
    rnn_units = 1024         # Number of LSTM units
    EPOCHS = 20              # Number of training epochs

    # 6. Build the LSTM Model
    print("\nBuilding the LSTM model...")
    model = models.Sequential([
        layers.Embedding(vocab_size, embedding_dim, batch_input_shape=[BATCH_SIZE, None]),
        layers.LSTM(rnn_units, return_sequences=True),
        layers.Dense(vocab_size)
    ])

    model.summary()

    # 7. Define the Loss Function
    def loss(labels, logits):
        return tf.keras.losses.sparse_categorical_crossentropy(labels, logits, from_logits=True)

    model.compile(optimizer='adam', loss=loss)

    # 8. Configure Checkpoints
    print("\nConfiguring checkpoints...")
    checkpoint_dir = './training_checkpoints'
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_prefix = os.path.join(checkpoint_dir, "ckpt_{epoch}")

    checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_prefix,
        save_weights_only=True
    )

    # 9. Train the Model
    print("\nStarting training...")
    history = model.fit(
        dataset,
        epochs=EPOCHS,
        callbacks=[checkpoint_callback]
    )

    # 10. Plot Training Loss
    print("\nPlotting training loss...")
    plt.figure(figsize=(8,6))
    plt.plot(history.history['loss'], label='Training Loss')
    plt.title('Training Loss Over Epochs')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.show()

    # 11. Build the Inference Model
    print("\nBuilding the inference model...")
    inference_model = models.Sequential([
        layers.Embedding(vocab_size, embedding_dim, batch_input_shape=[1, None]),
        layers.LSTM(rnn_units, return_sequences=True),
        layers.Dense(vocab_size)
    ])

    # Load the trained weights
    latest_checkpoint = tf.train.latest_checkpoint(checkpoint_dir)
    inference_model.load_weights(latest_checkpoint)
    inference_model.build(tf.TensorShape([1, None]))
    inference_model.summary()

    # 12. Define Text Generation Function
    def generate_text(model, start_string, num_generate=1000, temperature=1.0):
        """
        Generates text using the trained model.

        Args:
            model (tf.keras.Model): Trained LSTM model.
            start_string (str): Initial text to start the generation.
            num_generate (int): Number of characters to generate.
            temperature (float): Controls the randomness of predictions.

        Returns:
            str: Generated text.
        """
        # Converting start string to numbers (vectorizing)
        input_indices = [char2idx[c] for c in start_string]
        input_indices = tf.expand_dims(input_indices, 0)

        text_generated = []

        # Reset the model states
        model.reset_states()
        for _ in range(num_generate):
            predictions = model(input_indices)
            # Remove the batch dimension
            predictions = tf.squeeze(predictions, 0)

            # Using a categorical distribution to predict the character returned by the model
            predictions = predictions / temperature
            predicted_id = tf.random.categorical(predictions, num_samples=1)[-1, 0].numpy()

            # Pass the predicted character as the next input to the model
            input_indices = tf.expand_dims([predicted_id], 0)

            text_generated.append(idx2char[predicted_id])

        return start_string + ''.join(text_generated)

    # 13. Generate Sample Text
    print("\nGenerating text...")
    start_string = "ROMEO: "
    generated_text = generate_text(inference_model, start_string, num_generate=1000, temperature=0.5)
    print("\nGenerated Text:\n")
    print(generated_text)

if __name__ == "__main__":
    main()


Length of text: 1115394 characters

Preprocessing the text...
Unique characters: 65
Total characters in text: 1115394

Creating training sequences...

Building the LSTM model...


ValueError: Unrecognized keyword arguments passed to Embedding: {'batch_input_shape': [64, None]}